<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/SmolVLM_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate num2words av
!pip install -q decord || echo "decord not available, will use av instead"

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
).to("cuda")

print("Model loaded")

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

Model loaded


In [3]:
import os, gc, subprocess, torch

PROMPT = "Describe what is happening in this video in detail."  # one prompt for all videos
MAX_SECONDS = 30   # only first 30 sec of each video is used (lower this if you still get OOM)

def shrink_video(path, out="small.mp4"):
    if os.path.exists(out):
        os.remove(out)
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", path,
        "-t", str(MAX_SECONDS),
        "-vf", "fps=1,scale='min(384,iw)':-2",
        "-an", out
    ], check=True)
    return out

def describe_video(path, max_new_tokens=250):
    small = shrink_video(path)
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "path": small},
            {"type": "text", "text": PROMPT},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    print("Input tokens:", inputs["input_ids"].shape[-1])  # should be a few thousand, not tens of thousands

    with torch.no_grad():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)

    text = processor.batch_decode(output, skip_special_tokens=True)[0]
    del inputs, output
    gc.collect(); torch.cuda.empty_cache()
    return text.split("Assistant:")[-1].strip()

In [5]:
video_path="dog.mp4"
print(describe_video(video_path))

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Input tokens: 1083
The video features a dog, likely a Border Collie, standing on a grassy bank near a body of water. The dog appears to be in the midst of a playful or energetic activity, possibly running or jumping into the water. The dog's fur is wet, indicating that it has been in the water. The lighting in the video is bright, suggesting it was taken during the day. The dog's tail is raised, and it appears to be looking towards the camera, possibly as a sign of curiosity or excitement. The water is calm, with small ripples visible on the surface. The grassy bank is uneven, with some grass growing taller than the others. The dog's movements are fluid and dynamic, suggesting it is enjoying the activity.


In [4]:
print(describe_video("movie_clip1.mp4"))

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] LANCZOS resample requires torchvision >= 0.27 and processing on CPU; it is not supported on CUDA or for torchvision < 0.27. Falling back to BICUBIC which approximates LANCZOS. To match the original model exactly, upgrade torchvision and move the tensors to CPU before resizing, or use a PIL backend image processor instead.


Input tokens: 2869
The video opens with a view of a richly decorated room, featuring a large portrait of a woman in a pink dress, flanked by a vase of flowers and several framed photographs. The room is adorned with dark wood paneling, a large chandelier, and curtains. Two women are present, one standing and the other seated at a table with a lamp and a book. The standing woman is engaged in conversation with the seated woman, who is holding a notepad and pen. The scene then shifts to a different part of the room, where a woman with long curly hair is seen walking away from the camera, passing by a bookshelf and a desk with various items on it. The room's decor remains consistent throughout, with the same portrait, flowers, and framed photographs. The video continues to focus on the same richly decorated room, with the woman in the black coat and purple scarf standing and talking to the woman in the blue plaid skirt. The standing woman is holding a notepad and pen, while the seated wom

In [6]:
print(describe_video("movie_hindi.mp4"))

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Input tokens: 1083
The video starts with a woman in a black tank top with a heart design, light blue denim shorts, and a headband, standing in a store. She is holding a shopping bag and appears to be looking around. The scene then shifts to a different setting where the same woman is now wearing a black leather jacket and a colorful scarf, standing in a crowded street. The next scene shows her in a different outfit, wearing a black leather jacket and a red and blue scarf, standing in a different location. The video then transitions to a snowy outdoor setting where the woman is lying on the ground, surrounded by people. The final scenes show her in various outfits, including a blue and orange outfit, a red and orange outfit, and a white outfit, in different settings, including a crowded indoor area and a festive outdoor setting.
